# NB_00_Z_ENGINEERING_CONTEXT

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thinkthoughts/sensors-becker/blob/main/notebooks/NB_00_Z_ENGINEERING_CONTEXT.ipynb)

This notebook introduces the repository's initial (00) engineering artifact bundle.


In [ ]:
NOTEBOOK_ID = "NB_00_Z_ENGINEERING_CONTEXT"
NOTEBOOK_FILENAME = f"{NOTEBOOK_ID}.ipynb"
NOTEBOOK_VERSION = "1.0.0"
ENGINEERING_STAGE = "Z"
RELEASE_FILENAME = f"{NOTEBOOK_ID}.zip"

{
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "release_filename": RELEASE_FILENAME,
}


## Initialize Notebook Runtime

This operation prepares the repository environment used to generate the bundle.


In [ ]:
from __future__ import annotations

import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path


REPOSITORY_URL = "https://github.com/thinkthoughts/sensors-becker.git"
COLAB_REPOSITORY_ROOT = Path("/content/sensors-becker")


def install_colab_repository() -> Path:
    """Clone and install the repository in a fresh Colab runtime."""

    if COLAB_REPOSITORY_ROOT.exists():
        shutil.rmtree(COLAB_REPOSITORY_ROOT)

    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            REPOSITORY_URL,
            str(COLAB_REPOSITORY_ROOT),
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            str(COLAB_REPOSITORY_ROOT),
        ],
        check=True,
    )

    src_dir = COLAB_REPOSITORY_ROOT / "src"

    if str(src_dir) not in sys.path:
        sys.path.insert(0, str(src_dir))

    importlib.invalidate_caches()
    os.chdir(COLAB_REPOSITORY_ROOT)

    return COLAB_REPOSITORY_ROOT


try:
    import sensors_becker
except ModuleNotFoundError:
    repository_root = install_colab_repository()
    import sensors_becker
else:
    repository_root = Path(sensors_becker.__file__).resolve().parents[2]


if not (repository_root / "pyproject.toml").exists():
    raise FileNotFoundError(
        f"Repository root does not contain pyproject.toml: {repository_root}"
    )

from sensors_becker import initialize_notebook


runtime = initialize_notebook(
    start=repository_root,
    environment=(
        "google-colab"
        if repository_root == COLAB_REPOSITORY_ROOT
        else "repository-runtime"
    ),
)
context = runtime.context

print(f"Environment: {runtime.environment}")
print(f"Package: {Path(sensors_becker.__file__).resolve()}")
print(f"Repository root: {runtime.repository_root}")


## Validate Engineering Context

This operation validates the repository engineering context before artifact generation.


In [ ]:
runtime.validate()
print("Engineering context validation: PASSED")


## Prepare Notebook Bundle

This operation generates the source outputs and prepares the portable bundle directory.


In [ ]:
from __future__ import annotations

import json
import shutil
import zipfile
from datetime import date
from hashlib import sha256
from pathlib import Path

from IPython.display import Image, Markdown, display


def sha256_for_path(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


bundle_directory = runtime.paths.outputs / "releases" / NOTEBOOK_ID
bundle_directory.mkdir(parents=True, exist_ok=True)

# Remove files from a prior run so verification reflects this execution.
for prior_path in bundle_directory.iterdir():
    if prior_path.is_file():
        prior_path.unlink()

# Generate the repository's existing canonical source outputs.
json_source_path, yaml_source_path = runtime.export_context()
object_figure_source_path, cycle_figure_source_path = runtime.export_figures()

print(f"Bundle directory: {runtime.relative_path(bundle_directory)}")


## First (A)

This notebook introduces

**00_A_leading_trail.png**


In [ ]:
artifact_a_path = bundle_directory / "00_A_leading_trail.png"
shutil.copy2(cycle_figure_source_path, artifact_a_path)
display(Image(filename=str(artifact_a_path)))
print(runtime.relative_path(artifact_a_path))


## Second (B)

This notebook introduces

**00_B_sensor_trail.png**


In [ ]:
artifact_b_path = bundle_directory / "00_B_sensor_trail.png"
shutil.copy2(object_figure_source_path, artifact_b_path)
display(Image(filename=str(artifact_b_path)))
print(runtime.relative_path(artifact_b_path))


## Third (C)

This notebook introduces

**00_C_sensors-becker.md**


In [ ]:
artifact_c_path = bundle_directory / "00_C_sensors-becker.md"

context_markdown = f"""# sensors-becker

## Notebook

{NOTEBOOK_ID}

## Engineering Stage

{ENGINEERING_STAGE}

## Status

{context.status}

## Engineering Object

{context.engineering_object}

## Current Specification

{context.current_specification}

## Sensor Trail

""" + "\n".join(f"- {item}" for item in context.object_sequence) + """

## Engineering Paths

""" + "\n".join(f"- {path}" for path in context.engineering_paths) + """

## Measured Engineering States

""" + "\n".join(f"- {state}" for state in context.measured_engineering_states) + f"""

## Engineering Constraints

{chr(10).join(f"- {item}" for item in context.engineering_constraints) or "- Awaiting specification"}

## Engineering Refinements

{chr(10).join(f"- {item}" for item in context.engineering_refinements) or "- Awaiting specification"}

## Leading Specification

{context.leading_specification}

---

*{context.footer}*
"""

artifact_c_path.write_text(context_markdown, encoding="utf-8")
display(Markdown(context_markdown))
print(runtime.relative_path(artifact_c_path))


## Fourth (D)

This notebook introduces

**00_D_sensors-becker.yaml**


In [ ]:
artifact_d_path = bundle_directory / "00_D_sensors-becker.yaml"
shutil.copy2(yaml_source_path, artifact_d_path)

yaml_preview = artifact_d_path.read_text(encoding="utf-8")
print(yaml_preview)
print(runtime.relative_path(artifact_d_path))


## Fifth (E)

This notebook introduces

**00_E_sensors-becker.json**


In [ ]:
artifact_e_path = bundle_directory / "00_E_sensors-becker.json"
shutil.copy2(json_source_path, artifact_e_path)

with artifact_e_path.open(encoding="utf-8") as handle:
    json_preview = json.load(handle)

json_preview


## Record Notebook Bundle

The following records identify and verify the portable artifact bundle.


In [ ]:
artifact_f_path = bundle_directory / "00_F_README.md"
artifact_g_path = bundle_directory / "00_G_notebook_metadata.json"
artifact_h_path = bundle_directory / "00_H_manifest.json"

primary_artifact_paths = [
    artifact_a_path,
    artifact_b_path,
    artifact_c_path,
    artifact_d_path,
    artifact_e_path,
]

release_metadata = {
    "artifact_type": "notebook_bundle",
    "notebook_id": NOTEBOOK_ID,
    "notebook_filename": NOTEBOOK_FILENAME,
    "notebook_version": NOTEBOOK_VERSION,
    "repository": context.repository,
    "engineering_stage": ENGINEERING_STAGE,
    "engineering_object": context.engineering_object,
    "current_specification": context.current_specification,
    "runtime_environment": runtime.environment,
    "generated": date.today().isoformat(),
}

artifact_g_path.write_text(
    json.dumps(release_metadata, indent=2) + "\n",
    encoding="utf-8",
)

bundle_readme = f"""# {NOTEBOOK_ID}

This bundle contains the repository's initial (00) engineering artifacts in portable reading order.

## Artifacts

""" + "\n".join(f"- {path.name}" for path in primary_artifact_paths) + f"""
- {artifact_f_path.name}
- {artifact_g_path.name}
- {artifact_h_path.name}

## Repository

{context.repository}

## Engineering Object

{context.engineering_object}

## Current Specification

{context.current_specification}

---

*{context.footer}*
"""

artifact_f_path.write_text(bundle_readme, encoding="utf-8")

manifest_input_paths = primary_artifact_paths + [
    artifact_f_path,
    artifact_g_path,
]

manifest = {
    "release_filename": RELEASE_FILENAME,
    "repository": context.repository,
    "notebook_id": NOTEBOOK_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "engineering_stage": ENGINEERING_STAGE,
    "generated": date.today().isoformat(),
    "artifacts": [
        {
            "filename": path.name,
            "bundle_order": path.name.split("_", 2)[1],
            "size_bytes": path.stat().st_size,
            "sha256": sha256_for_path(path),
            "status": "verified",
        }
        for path in manifest_input_paths
    ],
}

artifact_h_path.write_text(
    json.dumps(manifest, indent=2) + "\n",
    encoding="utf-8",
)

for path in (artifact_f_path, artifact_g_path, artifact_h_path):
    print(runtime.relative_path(path))


## Verify Notebook Bundle

This operation verifies the complete A–H artifact sequence.


In [ ]:
bundle_artifact_paths = [
    artifact_a_path,
    artifact_b_path,
    artifact_c_path,
    artifact_d_path,
    artifact_e_path,
    artifact_f_path,
    artifact_g_path,
    artifact_h_path,
]

expected_names = [
    "00_A_leading_trail.png",
    "00_B_sensor_trail.png",
    "00_C_sensors-becker.md",
    "00_D_sensors-becker.yaml",
    "00_E_sensors-becker.json",
    "00_F_README.md",
    "00_G_notebook_metadata.json",
    "00_H_manifest.json",
]

actual_names = [path.name for path in bundle_artifact_paths]

assert actual_names == expected_names
assert all(path.exists() and path.stat().st_size > 0 for path in bundle_artifact_paths)

for path in bundle_artifact_paths:
    print(f"✓ {path.name} ({path.stat().st_size} bytes)")


## Package Notebook Bundle

This operation packages the verified artifacts without changing their portable reading order.


In [ ]:
release_directory = runtime.paths.outputs / "releases"
release_directory.mkdir(parents=True, exist_ok=True)
release_path = release_directory / RELEASE_FILENAME

with zipfile.ZipFile(release_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_artifact_paths:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(release_path) as archive:
    assert archive.namelist() == expected_names

assert release_path.stat().st_size > 0

print(
    f"✓ {runtime.relative_path(release_path)} "
    f"({release_path.stat().st_size} bytes)"
)


## Download Notebook Bundle

In Google Colab, this operation downloads the ZIP. Elsewhere, it prints the saved path.


In [ ]:
if runtime.environment == "google-colab":
    from google.colab import files

    files.download(str(release_path))
else:
    print(release_path)


*Admissible generalizations trail leading specifications.*
